In [0]:
# IMPORTANTE: ESTE COMANDO SÓ E EXECUTADO EM DESENVOLVIMENTO EM PRODUÇÃO SERÁ VIA JOB E A CONFIGURAÇÃO ESTÃO NO ARQUIVO resources/job.yml ou outros arquivo que será executado em produção
%uv sync

In [0]:
# %python
# # %pip install xgboost
# # %pip install xgboost shap
# %pip install -r ../../requirements.txt  -qqq

# dbutils.library.restartPython() 

In [0]:
%python
# Databricks notebook source
# =============================================================================
# 05_monitoramento_drift_concept.py
# -----------------------------------------------------------------------------
# Fase de Monitoramento do case "Risco de Crédito" (pós-deploy, item de
# Lakehouse Monitoring / MLOps do roadmap do README.md)
#
# O que este notebook faz:
#   1. Carrega o modelo @champion registrado no Unity Catalog (notebook 04)
#   2. Para cada novo batch de dados que chega (produção real OU simulação):
#        a) Calcula PSI (Population Stability Index) por feature vs. a base de
#           treino -> detecta DATA DRIFT (covariate shift)
#        b) Aplica o modelo congelado e calcula AUC-ROC, KS-statistic e
#           Recall@precisão-alvo -> detecta CONCEPT DRIFT (quando a relação
#           feature->target muda e o modelo passa a errar)
#   3. Registra as métricas de cada batch como uma run de MLflow (série
#      temporal de monitoramento, não um novo modelo)
#   4. Aplica thresholds de alerta e sinaliza quando o modelo precisa de
#      retreino
#
# Modo SIMULAÇÃO (SIMULATION_MODE = True):
#   Como o projeto ainda não tem cadência real de chegada de novos dados,
#   este modo GERA batches sintéticos a partir da distribuição de treino,
#   com deslocamento de distribuição (data drift) e mudança da relação
#   feature->target (concept drift) configuráveis — permite validar a lógica
#   de monitoramento e os thresholds de alerta ANTES de ter dados reais.
#   Quando os dados reais começarem a chegar, troque para False e aponte
#   NOVA_TABELA_BATCH para a tabela de produção.
#
# Por que PSI e não outra métrica de drift:
#   PSI é o padrão de mercado em risco de crédito (Basel/SR 11-7) por ser
#   interpretável e ter thresholds consolidados: <0.10 sem drift relevante,
#   0.10-0.25 drift moderado (atenção), >0.25 drift severo (ação).
# =============================================================================

# COMMAND ----------
# =========================
# 0. CONFIGURAÇÃO
# =========================
import mlflow
import mlflow.xgboost
import mlflow.sklearn
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score, precision_recall_curve
from mlflow.tracking import MlflowClient
import logging

logging.getLogger("mlflow").setLevel(logging.ERROR)
mlflow.set_registry_uri("databricks-uc")

CATALOG = "credito_prd"
SCHEMA = "gold"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.credito_risco_score"
NOVA_TABELA_BATCH = f"{CATALOG}.{SCHEMA}.fct_credit_profile_novos"  # tabela de produção (quando SIMULATION_MODE=False)
MONITORING_EXPERIMENT = "/Shared/credito_risco_monitoramento"

PRECISAO_ALVO = 0.30
FEATURE_COLS = [
    "age", "num_dependents", "monthly_income", "debt_ratio", "revolving_utilization",
    "num_open_credit_lines", "num_real_estate_loans", "num_times_30_59_days_late",
    "num_times_60_89_days_late", "num_times_90_days_late", "total_delinquency_events",
]
TARGET_COL = "target_dlq_2yrs"

# --- Thresholds de alerta (documentados, ajustar com a área de risco/negócio) ---
PSI_MODERADO = 0.10
PSI_SEVERO = 0.25
QUEDA_AUC_ALERTA = 0.03   # queda absoluta de AUC-ROC vs. baseline que dispara alerta de concept drift

# --- Toggle de simulação ---
SIMULATION_MODE = True
N_MESES_SIMULADOS = 10

mlflow.set_experiment(MONITORING_EXPERIMENT)

# COMMAND ----------
# =========================
# 1. CARREGA O CHAMPION
# =========================
client = MlflowClient()
versao_champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
auc_baseline = float(client.get_model_version(
    MODEL_NAME, versao_champion.version).tags.get("auc_roc", "0"))

# Carrega pelo FLAVOR NATIVO (não mlflow.pyfunc) com base na tag 'algoritmo'
# gravada pelo notebook 04. Dois motivos, confirmados em teste local antes
# de subir esta versão:
#   1. mlflow.pyfunc.load_model().predict() enforça o schema de forma rígida
#      e quebra (MlflowException) sempre que o dtype do batch difere do
#      dtype da assinatura — mesmo com os mesmos valores, mesmo alinhando
#      manualmente para float64.
#   2. Mais importante: para um XGBClassifier/pipeline sklearn, o pyfunc
#      .predict() retorna RÓTULO (0/1), não probabilidade — o que corrompe
#      silenciosamente AUC-ROC/KS/recall sem lançar nenhum erro. Carregando
#      o flavor nativo e chamando .predict_proba() diretamente evitamos os
#      dois problemas de uma vez.
algoritmo_champion = client.get_model_version(
    MODEL_NAME, versao_champion.version).tags.get("algoritmo", "xgboost")
if algoritmo_champion == "xgboost":
    modelo_champion = mlflow.xgboost.load_model(f"models:/{MODEL_NAME}@champion")
else:
    modelo_champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

print(f"Champion carregado: versão {versao_champion.version} ({algoritmo_champion}) | "
      f"AUC-ROC baseline: {auc_baseline:.4f}")

# COMMAND ----------
# =========================
# 2. FUNÇÕES DE MONITORAMENTO — reaproveitáveis para qualquer batch
# =========================
def calcular_psi(esperado, atual, bins=10):
    quebras = np.percentile(esperado, np.linspace(0, 100, bins + 1))
    quebras[0], quebras[-1] = -np.inf, np.inf
    quebras = np.unique(quebras)
    freq_esperada = np.histogram(esperado, bins=quebras)[0] / len(esperado)
    freq_atual = np.histogram(atual, bins=quebras)[0] / len(atual)
    freq_esperada = np.clip(freq_esperada, 1e-4, None)
    freq_atual = np.clip(freq_atual, 1e-4, None)
    return float(np.sum((freq_atual - freq_esperada) * np.log(freq_atual / freq_esperada)))


def alinhar_colunas_com_treino(df_batch, colunas_referencia):
    """Garante apenas a ORDEM das colunas igual à do treino (o dtype não
    precisa mais ser forçado: o flavor nativo do XGBoost/sklearn não enforça
    schema como o pyfunc fazia, e o XGBoost lida nativamente com int/float
    misturados e com NaN)."""
    return df_batch[colunas_referencia]


def calcular_ks(y_true, y_proba):
    return ks_2samp(y_proba[y_true == 1], y_proba[y_true == 0]).statistic


def recall_no_precision_alvo(y_true, y_proba, precisao_alvo=PRECISAO_ALVO):
    precisao, recall, _ = precision_recall_curve(y_true, y_proba)
    candidatos = recall[precisao >= precisao_alvo]
    return float(candidatos.max()) if len(candidatos) else 0.0


def classificar_psi(valor):
    if valor >= PSI_SEVERO:
        return "SEVERO"
    if valor >= PSI_MODERADO:
        return "MODERADO"
    return "OK"


def avaliar_batch(df_batch, df_referencia, modelo, run_name, auc_referencia=None):
    """Roda o ciclo completo de monitoramento para UM batch e loga no MLflow.
    Requer que df_batch já tenha o TARGET_COL disponível (rótulo com atraso
    conhecido de maturação, conforme a política de negócio do case).
    `auc_referencia`: AUC contra a qual medir degradação. Em produção real
    (SIMULATION_MODE=False), None cai no auc_baseline real do notebook 04 —
    correto, pois o modelo foi de fato calibrado para essa distribuição. Em
    simulação, é OBRIGATÓRIO passar a AUC do champion medida na própria
    df_referencia sintética (ver bloco 3) — comparar contra o baseline real
    faria o alerta disparar em TODOS os meses simulados, inclusive nos sem
    drift nenhum, porque o modelo real nunca foi calibrado para a relação
    feature->target sintética que este script inventa."""
    X_batch = alinhar_colunas_com_treino(df_batch[FEATURE_COLS], FEATURE_COLS)
    y_batch = df_batch[TARGET_COL]
    proba = modelo.predict_proba(X_batch)[:, 1]
    ref_auc = auc_referencia if auc_referencia is not None else auc_baseline

    metricas = {
        "auc_roc": float(roc_auc_score(y_batch, proba)),
        "ks_statistic": float(calcular_ks(y_batch.values, proba)),
        f"recall_em_{int(PRECISAO_ALVO*100)}pct_precisao": recall_no_precision_alvo(y_batch.values, proba),
        "taxa_positiva_batch": float(y_batch.mean()),
    }
    metricas["auc_referencia_usada"] = float(ref_auc)
    metricas["queda_auc_vs_baseline"] = ref_auc - metricas["auc_roc"]

    psi_por_feature = {
        feat: calcular_psi(df_referencia[feat].values, X_batch[feat].values)
        for feat in FEATURE_COLS
    }

    with mlflow.start_run(run_name=run_name):
        for chave, valor in metricas.items():
            mlflow.log_metric(chave, valor)
        for feat, valor in psi_por_feature.items():
            mlflow.log_metric(f"psi_{feat}", valor)
        mlflow.log_param("model_version_avaliada", versao_champion.version)

    # --- Alertas ---
    alertas = []
    psi_severo = {f: v for f, v in psi_por_feature.items() if classificar_psi(v) == "SEVERO"}
    psi_moderado = {f: v for f, v in psi_por_feature.items() if classificar_psi(v) == "MODERADO"}
    if psi_severo:
        alertas.append(f"[DATA DRIFT SEVERO] Features com PSI>={PSI_SEVERO}: {psi_severo}")
    elif psi_moderado:
        alertas.append(f"[DATA DRIFT MODERADO] Features com PSI>={PSI_MODERADO}: {psi_moderado}")
    if metricas["queda_auc_vs_baseline"] >= QUEDA_AUC_ALERTA:
        alertas.append(
            f"[CONCEPT DRIFT / DEGRADAÇÃO] AUC-ROC caiu {metricas['queda_auc_vs_baseline']:.4f} "
            f"vs. referência ({ref_auc:.4f} -> {metricas['auc_roc']:.4f}) — avaliar retreino."
        )
    for alerta in alertas:
        print(alerta)
    if not alertas:
        print(f"[{run_name}] OK — sem drift relevante. AUC-ROC: {metricas['auc_roc']:.4f}")

    return {**metricas, **{f"psi_{k}": v for k, v in psi_por_feature.items()}, "alertas": alertas}


def gerar_batch_simulado(n, deslocamento_drift, concept_drift, seed):
    """Gerador sintético usado tanto para a REFERÊNCIA (mês 0, sem
    deslocamento nem concept drift) quanto para os batches simulados dos
    meses seguintes. É ESSENCIAL que referência e batches venham do MESMO
    gerador: comparar uma referência real (tabelas de produção) contra
    batches sintéticos mediria a diferença entre "dado real" e "dado
    inventado", não drift de verdade — o PSI ficaria alto em TODAS as
    features desde o primeiro mês, inclusive nas que não deveriam mudar.

    Intensidade do concept_drift (ajustada após teste real): a primeira
    versão só invertia o papel de monthly_income/num_open_credit_lines, que
    o champion real provavelmente pondera pouco — o alerta de concept drift
    nunca disparou. Esta versão reduz drasticamente o peso de debt_ratio e
    revolving_utilization (de 2.2/2.6 para 0.7/0.8), que SÃO as variáveis
    dominantes em qualquer modelo de crédito -- um teste de estresse mais
    realista para validar que o alerta dispara quando a mudança é
    materialmente relevante."""
    r = np.random.default_rng(seed)
    age = r.normal(45, 12, n).clip(21, 90)
    monthly_income = r.lognormal(8.6, 0.6, n)
    debt_ratio = (r.gamma(2.0, 0.25, n) + deslocamento_drift).clip(0, 5)
    revolving_utilization = (r.beta(2, 5, n) + deslocamento_drift * 0.6).clip(0, 1.5)
    num_times_30_59 = r.poisson(0.35 + deslocamento_drift * 0.5, n).clip(0, 10)
    num_times_60_89 = r.poisson(0.12 + deslocamento_drift * 0.3, n).clip(0, 10)
    num_times_90 = r.poisson(0.08 + deslocamento_drift * 0.3, n).clip(0, 10)
    num_open_credit_lines = r.poisson(8, n).clip(0, 30)

    coef_renda = -0.00002 if not concept_drift else 0.00003
    coef_linhas = 0.0 if not concept_drift else 0.18
    logit = (
        (-3.6 if not concept_drift else -2.4)
        + (2.2 if not concept_drift else 0.7) * debt_ratio
        + (2.6 if not concept_drift else 0.8) * revolving_utilization
        + 0.55 * num_times_30_59 + 0.85 * num_times_60_89 + 1.05 * num_times_90
        + coef_linhas * num_open_credit_lines + coef_renda * monthly_income - 0.01 * age
    )
    target = r.binomial(1, 1 / (1 + np.exp(-logit)))
    return pd.DataFrame({
        "age": age, "num_dependents": r.poisson(0.9, n).clip(0, 8), "monthly_income": monthly_income,
        "debt_ratio": debt_ratio, "revolving_utilization": revolving_utilization,
        "num_open_credit_lines": num_open_credit_lines, "num_real_estate_loans": r.poisson(1.0, n).clip(0, 6),
        "num_times_30_59_days_late": num_times_30_59, "num_times_60_89_days_late": num_times_60_89,
        "num_times_90_days_late": num_times_90,
        "total_delinquency_events": num_times_30_59 + num_times_60_89 + num_times_90,
        TARGET_COL: target,
    })

# COMMAND ----------
# =========================
# 3. DISTRIBUIÇÃO DE REFERÊNCIA — depende do modo
# =========================
if SIMULATION_MODE:
    # Referência SINTÉTICA "mês 0" (sem deslocamento, sem concept drift), do
    # MESMO gerador usado nos batches simulados — garante que o PSI meça
    # drift de verdade, não a diferença entre dado real e dado inventado.
    df_referencia = gerar_batch_simulado(50_000, deslocamento_drift=0.0, concept_drift=False, seed=1)

    # AUC do champion medida NA PRÓPRIA REFERÊNCIA sintética (mês 0, sem
    # drift) — é ISSO que serve de comparador de degradação nos meses
    # seguintes, não o auc_baseline real. O modelo real nunca foi calibrado
    # para a relação feature->target sintética que este script inventa, então
    # comparar contra 0.8641 faria o alerta de concept drift disparar em
    # TODOS os meses, inclusive nos 4 primeiros sem nenhum drift injetado.
    X_referencia = alinhar_colunas_com_treino(df_referencia[FEATURE_COLS], FEATURE_COLS)
    proba_referencia = modelo_champion.predict_proba(X_referencia)[:, 1]
    auc_baseline_simulado = float(roc_auc_score(df_referencia[TARGET_COL], proba_referencia))

    print(f"[SIMULATION_MODE] Distribuição de referência gerada sinteticamente "
          f"(mesmo gerador dos batches, sem drift) — NÃO usa as tabelas reais do "
          f"Unity Catalog.\nAUC-ROC do champion nesta referência sintética: "
          f"{auc_baseline_simulado:.4f} (baseline real era {auc_baseline:.4f} — "
          f"NÃO comparável 1:1, a relação feature->target aqui é sintética). "
          f"Os meses seguintes são comparados contra {auc_baseline_simulado:.4f}.")
else:
    # Distribuição de referência real = a base de treino que gerou ESTA versão
    # específica do champion (notebook 04, bloco 7 — snapshot versionado pela
    # versão do modelo no Registry, não um nome fixo sobrescrito a cada treino).
    # Isso garante que, mesmo que um challenger mais novo já tenha sido
    # treinado sem virar champion, o monitoramento continua comparando o
    # champion em produção contra a base que efetivamente o treinou.
    # FALLBACK (só deve disparar para modelos treinados ANTES da versão 05
    # existir, sem snapshot salvo): recomputa o split com a MESMA query, seed
    # e test_size do notebook 04. Depende de a tabela Gold não ter mudado
    # desde o treino — tratar como solução temporária, não definitiva.
    tabela_snapshot_versionada = client.get_model_version(
        MODEL_NAME, versao_champion.version
    ).tags.get("snapshot_treino_tabela")

    try:
        if not tabela_snapshot_versionada:
            raise ValueError("Versão do champion não tem a tag 'snapshot_treino_tabela' "
                              "(treinada antes do notebook 04 gravar o snapshot).")
        df_referencia = spark.table(tabela_snapshot_versionada).toPandas()
        print(f"Distribuição de referência carregada de {tabela_snapshot_versionada}.")
    except Exception as e:
        print("[AVISO] Snapshot de treino não encontrado — recomputando o split "
              "a partir da mesma query do notebook 04 (fallback temporário). "
              f"Detalhe: {type(e).__name__}")
        from sklearn.model_selection import train_test_split

        FCT_TABLE, DIM_CUSTOMER_TABLE = "fct_credit_profile", "dim_customer"
        df_gold = spark.sql(f"""
            SELECT
                f.customer_id, c.age, c.num_dependents, f.monthly_income, f.debt_ratio,
                f.revolving_utilization, f.num_open_credit_lines, f.num_real_estate_loans,
                f.num_times_30_59_days_late, f.num_times_60_89_days_late,
                f.num_times_90_days_late, f.total_delinquency_events, f.{TARGET_COL}
            FROM {CATALOG}.{SCHEMA}.{FCT_TABLE} f
            JOIN {CATALOG}.{SCHEMA}.{DIM_CUSTOMER_TABLE} c ON f.customer_id = c.customer_id
        """).toPandas().dropna(subset=[TARGET_COL])
        df_gold[TARGET_COL] = df_gold[TARGET_COL].astype(int)
        df_referencia, _, _, _ = train_test_split(
            df_gold, df_gold[TARGET_COL], test_size=0.25, stratify=df_gold[TARGET_COL], random_state=42
        )

# COMMAND ----------
# =========================
# 4. MODO SIMULAÇÃO — roda os N meses sintéticos com drift crescente
# =========================
if SIMULATION_MODE:
    historico = []
    for mes in range(1, N_MESES_SIMULADOS + 1):
        if mes <= 4:
            deslocamento, concept = 0.0, False
        elif mes <= 7:
            deslocamento, concept = 0.06 * (mes - 4), False
        else:
            deslocamento, concept = 0.18 + 0.05 * (mes - 7), True

        df_batch_sim = gerar_batch_simulado(8_000, deslocamento, concept, seed=100 + mes)
        resultado = avaliar_batch(
            df_batch_sim, df_referencia, modelo_champion,
            run_name=f"monitor_mes_simulado_{mes:02d}",
            auc_referencia=auc_baseline_simulado,
        )
        resultado["mes"] = mes
        historico.append(resultado)

    df_historico = pd.DataFrame(historico)
    display(df_historico)  # noqa: F821 -- display() é global no runtime Databricks

# COMMAND ----------
# =========================
# 5. MODO PRODUÇÃO — quando SIMULATION_MODE=False, roda 1x por chegada de batch real
# =========================
if not SIMULATION_MODE:
    df_batch_real = spark.sql(f"""
        SELECT * FROM {NOVA_TABELA_BATCH}
        WHERE batch_date = current_date() - INTERVAL 1 DAY
          AND {TARGET_COL} IS NOT NULL   -- só entra no monitoramento quando o rótulo já maturou
    """).toPandas()

    if len(df_batch_real) == 0:
        print("Nenhum batch novo com rótulo maturado disponível para monitoramento hoje.")
    else:
        avaliar_batch(df_batch_real, df_referencia, modelo_champion, run_name=f"monitor_prod_{pd.Timestamp.now():%Y%m%d}")

# COMMAND ----------
# =========================
# 6. GATILHO DE RETREINO (regra, não julgamento manual)
# =========================
# Sugestão de política: se DOIS batches consecutivos disparam alerta de concept
# drift (queda de AUC >= QUEDA_AUC_ALERTA), abrir chamado de retreino automático
# do notebook 04 e pausar a promoção automática a @champion até validação humana.
print(f"\nThresholds ativos — PSI moderado: {PSI_MODERADO} | PSI severo: {PSI_SEVERO} | "
      f"Queda de AUC para alerta: {QUEDA_AUC_ALERTA}")
